In [1]:
import pandas as pd
import os
import numpy as np
import scipy as sp
import scipy.stats as sps
import matplotlib.pyplot as plt
import networkx as nx
from scipy.optimize import curve_fit
from scipy.stats import pearsonr
from scipy.stats import spearmanr,kendalltau
import math
import ruptures as rpt
import seaborn as sns
import matplotlib.cm as cm
import matplotlib.colors as colors

In [ ]:
ds = pd.DataFrame(pd.read_csv("dati.csv")) #upload and conversion to dataframe of the data

'Normalization and NaN remouval'
col_interessate = list(ds.columns[9:29]) + list(ds.columns[38:46])
df = pd.DataFrame() 

range_m3_3 = ['mood_down', 'mood_lonely', 'mood_anxious', 'mood_guilty'] 
range_1_7 = [col for col in col_interessate if col not in range_m3_3]

for col in range_m3_3:
    df[col] = (ds[col] - (-3)) / (3 - (-3))
    

for col in range_1_7:
    df[col] = (ds[col] - 1) / (7 - 1)
    
df['date'] = ds['date']
df['resptime_e'] = ds['resptime_e']

df = df.dropna()

'Set data_time = data + resptime_e as the index'
df=df.dropna(subset=['resptime_e']) 
df['date']=pd.to_datetime(df['date'],format="%d/%m/%y") 
df['resptime_e'] = df['resptime_e'].apply(lambda t: pd.Timedelta(hours=t.hour, minutes=t.minute, seconds=t.second))

#df['resptime_e']=pd.to_timedelta(df['resptime_e']) 
df['date_time'] = df['date'] + df['resptime_e'] 
df['date_time']=pd.to_datetime(df['date_time']) 
df=df.dropna(subset=['date_time']) 
df = df.dropna(subset=['date_time']).sort_values('date_time')
df=df.set_index('date_time') 

# DYNAMICAL GRAPHS STRUCTURE

In [ ]:
# Dynamical graphs creation with sensitivity analysis 
data = df.to_numpy()
df, N = data.shape
real_dates = df.index 

# first window choice
win1 = 40
step = 10
win_starts1 = np.arange(0, df - win1 + 1, step)
numW = len(win_starts1)
A1 = np.zeros((N, N, numW))
md1 = []
timeIndex_dates1 = []

for k, start in enumerate(win_starts1):
    idx = np.arange(start, start + win1)
    t_center = idx[-1]
    timeIndex_dates1.append(real_dates[t_center])
    
    Xw = data[idx, :]

    # Constant columns
    std_cols = Xw.std(axis=0)
    const_cols = std_cols == 0
    if np.any(const_cols):
        Xw[:, const_cols] += 1e-8 * np.random.randn(Xw.shape[0], np.sum(const_cols))

    # correlation matrix
    deltaW = np.diff(Xw, axis=0)
    C = np.abs(np.corrcoef(deltaW, rowvar=False))
    np.fill_diagonal(C, 0)
    
    C = np.nan_to_num(C)
    mean_strength = np.mean(C.sum(axis=0)) 
    md1.append(mean_strength)
    A1[:, :, k] = C

timeIndex_dates1 = np.array(timeIndex_dates1)

# change point detection
model = "l1"  
md_array1 = np.array(md1)
algo = rpt.BottomUp(model=model, min_size=2, jump=5).fit(md_array1) 
my_bkps = algo.predict(n_bkps=1)
cp1 = my_bkps[0] - 1 
cp_date1 = timeIndex_dates1[cp1]

# second window choice
win2 = 70
step = 10
win_starts2 = np.arange(0, Tlen - win2 + 1, step)
numW = len(win_starts2)
A2 = np.zeros((N, N, numW))
md2 = []
timeIndex_dates2 = []

for k, start in enumerate(win_starts2):
    idx = np.arange(start, start + win2)
    t_center = idx[-1]
    timeIndex_dates2.append(real_dates[t_center])
    
    Xw = data[idx, :]

    # constant columns
    std_cols = Xw.std(axis=0)
    const_cols = std_cols == 0
    if np.any(const_cols):
        Xw[:, const_cols] += 1e-8 * np.random.randn(Xw.shape[0], np.sum(const_cols))

    deltaW = np.diff(Xw, axis=0)
    C = np.abs(np.corrcoef(deltaW, rowvar=False))
    np.fill_diagonal(C, 0)
    
    C = np.nan_to_num(C)
    mean_strength = np.mean(C.sum(axis=0)) 
    md2.append(mean_strength)
    A2[:, :, k] = C


timeIndex_dates2 = np.array(timeIndex_dates2)

# change point detection
model = "l1"  
md_array2 = np.array(md2)
algo = rpt.BottomUp(model=model, min_size=2, jump=5).fit(md_array2) 
my_bkps = algo.predict(n_bkps=1)
cp2 = my_bkps[0] - 1 
cp_date2 = timeIndex_dates2[cp2]

# third window choice
win3 = 140
step = 10
win_starts3 = np.arange(0, Tlen - win3 + 1, step)
numW = len(win_starts3)
A3 = np.zeros((N, N, numW))
md3 = []
timeIndex_dates3 = []

for k, start in enumerate(win_starts3):
    idx = np.arange(start, start + win3)
    t_center = idx[-1]
    timeIndex_dates3.append(real_dates[t_center])
    
    Xw = data[idx, :]

    std_cols = Xw.std(axis=0)
    const_cols = std_cols == 0
    if np.any(const_cols):
        Xw[:, const_cols] += 1e-8 * np.random.randn(Xw.shape[0], np.sum(const_cols))

        deltaW = np.diff(Xw, axis=0)
    C = np.abs(np.corrcoef(deltaW, rowvar=False))
    np.fill_diagonal(C, 0)
    C = np.nan_to_num(C)
    mean_strength = np.mean(C.sum(axis=0)) 
    md3.append(mean_strength)
    A3[:, :, k] = C


timeIndex_dates3 = np.array(timeIndex_dates3)

# change point detection
model = "l1"  
md_array3 = np.array(md3)
algo = rpt.BottomUp(model=model, min_size=2, jump=5).fit(md_array3) 
my_bkps = algo.predict(n_bkps=1)
cp3 = my_bkps[0] - 1 
cp_date3 = timeIndex_dates3[cp3]


# fourth window choice
win4 = 210
step = 10
win_starts4 = np.arange(0, Tlen - win4 + 1, step)
numW = len(win_starts4)
A4 = np.zeros((N, N, numW))
md4 = []
timeIndex_dates4 = []

for k, start in enumerate(win_starts4):
    idx = np.arange(start, start + win4)
    t_center = idx[-1]
    timeIndex_dates4.append(real_dates[t_center])
    
    Xw = data[idx, :]

    std_cols = Xw.std(axis=0)
    const_cols = std_cols == 0
    if np.any(const_cols):
        Xw[:, const_cols] += 1e-8 * np.random.randn(Xw.shape[0], np.sum(const_cols))
    deltaW = np.diff(Xw, axis=0)
    C = np.abs(np.corrcoef(deltaW, rowvar=False))
    np.fill_diagonal(C, 0)
    
    C = np.nan_to_num(C)
    mean_strength = np.mean(C.sum(axis=0)) 
    md4.append(mean_strength)
    A4[:, :, k] = C


timeIndex_dates4 = np.array(timeIndex_dates4)

# change point detection
model = "l1"  
md_array4 = np.array(md4)
algo = rpt.BottomUp(model=model, min_size=2, jump=5).fit(md_array4) 
my_bkps = algo.predict(n_bkps=1)
cp4 = my_bkps[0] - 1 
cp_date4 = timeIndex_dates3[cp4]


# Plot 
plt.figure(figsize=(26, 12))

plt.subplot(2,2,1)

plt.plot(timeIndex_dates1, md1/max(md1), label='Average Degree (Strength)', linewidth=2)
plt.axvspan(timeIndex_dates1.min(), cp_date1, color='lightblue', alpha=0.3, label='Pre-transition')
plt.axvspan(cp_date1, timeIndex_dates1.max(), color='lightcoral', alpha=0.3, label='Post-transition')
plt.axvline(cp_date1, color='red', linestyle='--', label='Transition Point')
plt.ylabel('Average Degree', fontsize=25)
plt.legend(loc='upper left')
plt.title(f"window size w = {win1}", fontsize=35)
plt.grid(True, alpha=0.3)

plt.subplot(2,2,2)

plt.plot(timeIndex_dates2, md2/max(md2), label='Average Degree (Strength)', linewidth=2)
plt.axvspan(timeIndex_dates2.min(), cp_date2, color='lightblue', alpha=0.3, label='Pre-transition')
plt.axvspan(cp_date2, timeIndex_dates2.max(), color='lightcoral', alpha=0.3, label='Post-transition')
plt.axvline(cp_date2, color='red', linestyle='--', label='Transition Point')
plt.ylabel('Average Degree', fontsize=25)
plt.legend(loc='upper left')
plt.title(f"window size w = {win2}", fontsize=35)
plt.grid(True, alpha=0.3)

plt.subplot(2,2,3)

plt.plot(timeIndex_dates3, md3/max(md3), label='Average Degree (Strength)', linewidth=2)
plt.axvspan(timeIndex_dates3.min(), cp_date3, color='lightblue', alpha=0.3, label='Pre-transition')
plt.axvspan(cp_date3, timeIndex_dates3.max(), color='lightcoral', alpha=0.3, label='Post-transition')
plt.axvline(cp_date3, color='red', linestyle='--', label='Transition Point')
plt.xlabel('Date', fontsize=25)
plt.ylabel('Average Degree', fontsize=25)
plt.legend(loc='upper left')
plt.title(f"window size w = {win3}", fontsize=35)
plt.grid(True, alpha=0.3)

plt.subplot(2,2,4)

plt.plot(timeIndex_dates4, md4/max(md4), label='Average Degree (Strength)', linewidth=2)
plt.axvspan(timeIndex_dates4.min(), cp_date4, color='lightblue', alpha=0.3, label='Pre-transition')
plt.axvspan(cp_date4, timeIndex_dates4.max(), color='lightcoral', alpha=0.3, label='Post-transition')
plt.axvline(cp_date4, color='red', linestyle='--', label='Transition Point')
plt.xlabel('Date', fontsize=25)
plt.ylabel('Average Degree', fontsize=25)
plt.legend(loc='upper left')
plt.title(f"window size w = {win4}", fontsize=35)
plt.grid(True, alpha=0.3)

plt.show()

In [ ]:
# Elbow method for the window size choice

w_range = np.arange(20, 500, 10)
step = 10 

instability_scores = []

for w in w_range:
    win_starts = np.arange(0, Tlen - w + 1, step)
    md_w = []
    
    for start in win_starts:
        idx = np.arange(start, start + w)
        Xw = data[idx, :]
        
        std_cols = Xw.std(axis=0)
        const_cols = std_cols == 0
        if np.any(const_cols):
            Xw[:, const_cols] += 1e-8 * np.random.randn(Xw.shape[0], np.sum(const_cols))
            
        deltaW = np.diff(Xw, axis=0)
        C = np.abs(np.corrcoef(deltaW, rowvar=False))
        np.fill_diagonal(C, 0)
        C = np.nan_to_num(C)
        
        mean_strength = np.mean(C.sum(axis=0))
        md_w.append(mean_strength)
        
    # Standard deviation computation
    if len(md_w) > 1:
        instability=np.std(np.abs(md_w))/np.mean(md_w)
        
    else:
        instability = np.nan 
        
    instability_scores.append(instability)


# PLOT
plt.figure(figsize=(10, 6))
plt.plot(w_range, instability_scores, marker='o', linewidth=2, color='#2c3e50', markersize=8)

plt.xlabel('Window Size', fontsize=18)
plt.ylabel('Coefficient of Variation', fontsize=18)
plt.title('Structural Stability', fontsize=25, pad=15)
plt.grid(True, linestyle='--', alpha=0.6)

plt.xticks(np.arange(20, 500, 20), fontsize=12)
plt.yticks(fontsize=12)

plt.tight_layout()
plt.show()

In [ ]:
# LOVO analysis fixing the window choice w = 210
w = 210
step = 10

Tlen = len(df)
real_dates = df.index
columns = df.columns

win_starts = np.arange(0, Tlen - w + 1, step)
timeIndex_dates = []

for start in win_starts:
    idx = np.arange(start, start + w)
    t_center = idx[-1]
    timeIndex_dates.append(real_dates[t_center])
    
timeIndex_dates = np.array(timeIndex_dates)

lovo_cp_dates = {}

for dropped_col in columns:
    data_lovo = T_sel.drop(columns=[dropped_col]).to_numpy()
    
    md_lovo = []
    
    # compute again the dynamical network structure excluding the selected item
    for start in win_starts:
        idx = np.arange(start, start + w)
        Xw = data_lovo[idx, :]
        
        std_cols = Xw.std(axis=0)
        const_cols = std_cols == 0
        if np.any(const_cols):
            Xw[:, const_cols] += 1e-8 * np.random.randn(Xw.shape[0], np.sum(const_cols))
            
        deltaW = np.diff(Xw, axis=0)
        C = np.abs(np.corrcoef(deltaW, rowvar=False))
        np.fill_diagonal(C, 0)
        C = np.nan_to_num(C)
        
        # Average Degree
        mean_strength = np.mean(C.sum(axis=0))
        md_lovo.append(mean_strength)
        
    # Changepoint detection whitout the selected item
    md_array = np.array(md_lovo)
    algo = rpt.BottomUp(model="l1", min_size=2, jump=5).fit(md_array)
    my_bkps = algo.predict(n_bkps=1)
    
    cp_idx = my_bkps[0] - 1
    cp_date = timeIndex_dates[cp_idx]
    
    lovo_cp_dates[dropped_col] = cp_date
    print(f"Escluso: {dropped_col: <20} -> Data Changepoint: {cp_date.strftime('%Y-%m-%d')}")


# PLOT 


nodi = ['Relaxed', 'Down', 'Irritated', 'Satisfied', 'Lonely', 'Anxious', 'Enthusiastic', 'Suspicious', 
        'Cheerful', 'Guilty', 'Doubt', 'Strong', 'Restless', 'Agitated', 'Worry', 'Concentrated', 'Selflike', 
        'Ashemed', 'Selfdoubt', 'Handle', 'Hungry', 'Tired', 'Pain', 'Dizzy', 'Drymouth', 'Nauseous', 'Headache', 'Sleepy']

symptoms = list(lovo_cp_dates.keys())
dates = list(lovo_cp_dates.values())

median_date = pd.Series(dates).median()

plt.figure(figsize=(12, 8))


plt.scatter(dates, range(len(symptoms)), color='#2c3e50', s=100, zorder=3, label='LOVO Changepoint Date')

plt.axvline(median_date, color='red', linestyle='--', linewidth=2, zorder=2, label=f'Median Date ({median_date.strftime("%Y-%m-%d")})')

# Graphical improvements
plt.yticks(range(len(symptoms)), labels = nodi, fontsize=16)
plt.xticks(rotation = 60)
plt.xlabel("Date", fontsize=18)
plt.title("Leave-One-Variable-Out Sensitivity Analysis", fontsize=25, pad=15)

for i in range(len(symptoms)):
    plt.axhline(i, color='gray', linestyle=':', alpha=0.3, zorder=1)

plt.grid(axis='x', linestyle='--', alpha=0.6)
plt.legend(loc='upper right')
plt.tight_layout()

plt.show()

In [ ]:
# ADDITONAL SURROGATE ANALYSIS

# ramp function: x1 before ts, linear ramp ts->te, x2 after te
def ramp_model(t, x1, x2, ts, te):
    yhat = np.piecewise(
        t,
        [t < ts, (t >= ts) & (t <= te), t > te],
        [
            lambda t: x1,
            lambda t: x1 + (x2 - x1) * (t - ts) / (te - ts),
            lambda t: x2
        ]
    )
    return yhat

# Surrogate analysis
w = 210
step = 10
n_shuffles = 1000

data_orig = T_sel.to_numpy()
Tlen, N = data_orig.shape
real_dates = T_sel.index

# real trajectory
win_starts = np.arange(0, Tlen - w + 1, step)
md_real = []
timeIndex_dates = []

for start in win_starts:
    idx = np.arange(start, start + w)
    t_center = idx[-1]
    timeIndex_dates.append(real_dates[t_center])
    
    Xw = data_orig[idx, :]
    
    std_cols = Xw.std(axis=0)
    const_cols = std_cols == 0
    if np.any(const_cols):
        Xw[:, const_cols] += 1e-8 * np.random.randn(Xw.shape[0], np.sum(const_cols))
        
    deltaW = np.diff(Xw, axis=0)
    C = np.abs(np.corrcoef(deltaW, rowvar=False))
    np.fill_diagonal(C, 0)
    C = np.nan_to_num(C)
    
    md_real.append(np.mean(C.sum(axis=0)))

md_array_real = np.array(md_real)
timeIndex_dates = np.array(timeIndex_dates)

# real Changepoint 
algo_real = rpt.BottomUp(model="l1", min_size=2, jump=5).fit(md_array_real)
my_bkps_real = algo_real.predict(n_bkps=1)
cp_idx_real = my_bkps_real[0] - 1
cp_date_real = timeIndex_dates[cp_idx_real]

# Shuffled trajectories
all_shuffles_md = []

for i in range(n_shuffles):
    np.random.seed(42 + i)
    shuffled_idx = np.random.permutation(Tlen)
    data_shuff = data_orig[shuffled_idx, :]
    
    md_shuff = []
    for start in win_starts:
        idx = np.arange(start, start + w)
        Xw = data_shuff[idx, :]
        
        std_cols = Xw.std(axis=0)
        const_cols = std_cols == 0
        if np.any(const_cols):
            Xw[:, const_cols] += 1e-8 * np.random.randn(Xw.shape[0], np.sum(const_cols))
            
        deltaW = np.diff(Xw, axis=0)
        C = np.abs(np.corrcoef(deltaW, rowvar=False))
        np.fill_diagonal(C, 0)
        C = np.nan_to_num(C)
        
        md_shuff.append(np.mean(C.sum(axis=0)))
        
    all_shuffles_md.append(np.array(md_shuff))


tutte_le_traiettorie = np.concatenate([md_array_real] + all_shuffles_md)
g_min = np.min(tutte_le_traiettorie)
g_max = np.max(tutte_le_traiettorie)

# normalization
md_array_real_norm = (md_array_real - g_min) / (g_max - g_min)
all_shuffles_md_norm = [(shuff - g_min) / (g_max - g_min) for shuff in all_shuffles_md]

t_0 = timeIndex_dates[0]
t_days = np.array([(dt - t_0).total_seconds() / (24 * 3600) for dt in timeIndex_dates])
margin = max(2, len(t_days) // 10) # Margine di sicurezza per le stime iniziali p0

# ramp fitting  
algo_real = rpt.BottomUp(model="l1", min_size=2, jump=5).fit(md_array_real_norm)
cp_idx_real = algo_real.predict(n_bkps=1)[0] - 1

idx_s_r = max(0, cp_idx_real - margin)
idx_e_r = min(len(t_days) - 1, cp_idx_real + margin)

p0_real = [md_array_real_norm[idx_s_r], md_array_real_norm[idx_e_r], t_days[idx_s_r], t_days[idx_e_r]]
popt_real, _ = curve_fit(ramp_model, t_days, md_array_real_norm, p0=p0_real, maxfev=10000)

h_real = popt_real[1] - popt_real[0]         
slope_real = h_real / (popt_real[3] - popt_real[2]) if (popt_real[3] - popt_real[2]) != 0 else 0


h_shuff = []
slope_shuff = []


MIN_BOUNDS = [0.0, 0.0, 0.0, 0.0]
MAX_BOUNDS = [1.0, 1.0, t_days[-1], t_days[-1]]


popt_real, _ = curve_fit(
    ramp_model, t_days, md_array_real_norm, 
    p0=p0_real, 
    bounds=(MIN_BOUNDS, MAX_BOUNDS), 
    maxfev=10000
)

for i, y_shuff in enumerate(all_shuffles_md_norm):
    algo_shuff = rpt.BottomUp(model="l1", min_size=2, jump=5).fit(y_shuff)
    cp_idx_shuff = algo_shuff.predict(n_bkps=1)[0] - 1
    
    idx_s_s = max(0, cp_idx_shuff - margin)
    idx_e_s = min(len(t_days) - 1, cp_idx_shuff + margin)
    
    p0_shuff = [y_shuff[idx_s_s], y_shuff[idx_e_s], t_days[idx_s_s], t_days[idx_e_s]]
    
    try:
        popt_s, _ = curve_fit(ramp_model, t_days, y_shuff, p0=p0_shuff, bounds=(MIN_BOUNDS, MAX_BOUNDS),
                              maxfev=10000)
        h_s = popt_s[1] - popt_s[0]
        sl_s = h_s / (popt_s[3] - popt_s[2]) if (popt_s[3] - popt_s[2]) != 0 else 0
        
        h_shuff.append(h_s)
        slope_shuff.append(sl_s)
    except RuntimeError:
        h_shuff.append(np.nan)
        slope_shuff.append(np.nan)

h_shuff = np.array(h_shuff)[~np.isnan(h_shuff)]
slope_shuff = np.array(slope_shuff)[~np.isnan(slope_shuff)]

# PLOT
print("="*50)
print(f"{'RISULTATI DEL CONFRONTO STATISTICO':^50}")
print("="*50)
print(f"Detected data  -> Ramp High: {h_real:.4f} | Slope: {slope_real:.4f}")
print("-"*50)
print(f"AVERAGED SHUFFLED  -> Ramp High: {np.mean(h_shuff):.4f} | Slope: {np.mean(slope_shuff):.4f}")
print(f"Max SHUFFLED    -> Ramp High: {np.max(h_shuff):.4f} | Slope: {np.max(slope_shuff):.4f}")
print("="*50)

# Calcolo del percentile (quanti surrogati hanno un'altezza inferiore al dato reale)
p_val_height = np.sum(h_shuff >= h_real) / len(h_shuff)
print(f"Proporzione di dati Shuffled con altezza >= Reale: {p_val_height:.4f}")


# BOXPLOT
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Boxplot per le Altezze (Heights)
axes[0].boxplot(h_shuff, patch_artist=True, boxprops=dict(facecolor='lightgray', color='gray'))
axes[0].scatter(1, h_real, color='red', s=150, zorder=5, label='Empirical Data')
axes[0].set_xticklabels(['Shuffled Data'])
axes[0].set_ylabel('Ramp Height ($\Delta x$)', fontsize=12)
axes[0].set_title('Confronto Altezza Rampa', fontsize=14)
axes[0].grid(True, linestyle='--', alpha=0.5)
axes[0].legend()

# Boxplot per le Inclinazioni (Slopes)
axes[1].boxplot(slope_shuff, patch_artist=True, boxprops=dict(facecolor='lightgray', color='gray'))
axes[1].scatter(1, slope_real, color='red', s=150, zorder=5, label='Empirical Data')
axes[1].set_xticklabels(['Shuffled Data'])
axes[1].set_ylabel('Ramp Slope', fontsize=12)
axes[1].set_title('Confronto Inclinazione Rampa', fontsize=14)
axes[1].grid(True, linestyle='--', alpha=0.5)
axes[1].legend()

plt.tight_layout()
plt.show()


# SHAFFLED DATA PLOT
plt.figure(figsize=(12, 7))

for i, md_shuff_norm in enumerate(all_shuffles_md_norm):
    if i == 0:
        plt.plot(timeIndex_dates, md_shuff_norm, color='gray', alpha=0.3, linewidth=1.5, zorder=1, label=f'Surrogate Data Permutations)')
    else:
        plt.plot(timeIndex_dates, md_shuff_norm, color='gray', alpha=0.3, linewidth=1.5, zorder=1)

plt.plot(timeIndex_dates, md_array_real_norm, color='#2c3e50', linewidth=3, zorder=5, label='Empirical Data')

plt.axvline(cp_date_real, color='red', linestyle='--', linewidth=2, zorder=6, label=f'Detected Changepoint')

# plot improvement
plt.title('Surrogate Data', fontsize=25, pad=15)
plt.xlabel('Date', fontsize=18)
plt.ylabel('Normalized Average Degree', fontsize=18) 

plt.grid(True, linestyle='--', alpha=0.5, zorder=0)
plt.legend(loc='upper right', fontsize=11, framealpha=0.9)

plt.ylim(-0.05, 1.05) 

plt.gcf().autofmt_xdate()

plt.tight_layout()
plt.show()


# HEIGHTS ISTOGRAM
fig= plt.subplots(figsize=(14, 6))


plt.hist(h_shuff, bins=100, color='lightblue', edgecolor='lightblue', alpha=0.7, label='Shuffled Data')
plt.axvline(h_real, color='red', linestyle='--', linewidth=3, zorder=5, 
                label=f'Empirical Data ({h_real:.3f})')

plt.xlabel('Ramp High $\Delta x$', fontsize=18)
plt.title('Height distribution', fontsize=25)
plt.grid(True, linestyle='--', alpha=0.5)
plt.legend(loc='upper right')


plt.tight_layout()
plt.show()

# UTILITy FUNCTIONS

In [ ]:
def recurrence_grammar(R):
    
    T = R.shape[0]         # dimensione della serie
    s = np.arange(T)       # sequenza iniziale degli indici
    
    changed = True
    while changed:
        changed = False
        for i in range(T):
            for j in range(i):
                if R[i,j] == 1 and s[i] > s[j]:
                    s[i] = s[j]
                    changed = True
    return s

def compute_entropies(P):
    n = P.shape[0]
    
    if n > 2:
        # normalizza riga e colonna 0
        row0 = P[0, 1:] / np.sum(P[0, 1:]) if np.sum(P[0, 1:]) > 0 else np.ones(n-1)/(n-1)
        col0 = P[1:, 0] / np.sum(P[1:, 0]) if np.sum(P[1:, 0]) > 0 else np.ones(n-1)/(n-1)
        
        # calcola entropia con epsilon per evitare log(0)
        eps = 1e-12
        hr = -(1/np.log(n-1)) * np.sum(row0 * np.log(row0 + eps))
        hc = -(1/np.log(n-1)) * np.sum(col0 * np.log(col0 + eps))
    else:
        hr, hc = 0.0, 0.0
    
    return hr, hc


In [ ]:
df = df.set_index('date_time').sort_index()
for col in col_interessate:
    df[col + '7d'] = df.rolling('7D')[col].mean()
    df[col + '14d'] = df.rolling('14D')[col].mean()
    df[col + '21d'] = df.rolling('21D')[col].mean()
    df[col + '50d'] = df.rolling('50D')[col].mean()
    df[col + '100d'] = df.rolling('100D')[col].mean()

In [ ]:
results = []
for col in symptoms:
    sintomo = col + '50d' #'7d' '14d' '21d' '100d'
    x = df[sintomo].values  # array 1D
    T = len(x)
    D = np.abs(x[:, np.newaxis] - x[np.newaxis, :])

    # Epsilon range 
    eps_range = np.linspace(0.01, 0.1, 100)  
    # --- Inizialization ---
    best_utility = -np.inf
    best_n_states = None
    eps_star = None
    P_star = None
    utilities = [] 
    
    # --- Loop on epsilon ---
    for e in eps_range:
        # 1. recurrence plot
        R = (D < e).astype(int)
    
        # 2. metastates partition
        partitions = recurrence_grammar(R)
        
        
        unique_states, s_idx = np.unique(partitions, return_inverse=True)
        
        n_states = len(unique_states)
    
        # 3. transition matrix
        P_counts = np.zeros((n_states, n_states))
        for t in range(len(s_idx)-1):
            i = s_idx[t]
            j = s_idx[t+1]
            P_counts[i, j] += 1
        P = P_counts / (P_counts.sum(axis=1, keepdims=True) + 1e-12)  # normalizzazione righe
    
        # 4. entropies and utility
        hr, hc = compute_entropies(P)
        utility = (np.trace(P) + hr + hc) / (n_states + 2)
        utilities.append(utility)
        # 5. epsilon update
        if utility > best_utility:
            best_utility = utility
            eps_star = e
            P_star = P
            best_n_states = n_states

     # Save current results
    results.append({
        'Symptom': sintomo,
        'Optimal e': eps_star,
        'Best_utility': best_utility,
        'Number metastates':best_n_states
    })
    
    # --- Plot ---
    plt.figure(figsize=(12,5))
    plt.plot(eps_range, utilities, marker='o')
    plt.axvline(eps_star, color='red', linestyle='--', label=f'Optimal ε = {eps_star:.5f}')
    plt.ylim(-0.02,1.02)
    plt.tick_params(axis='both', labelsize=18)
    plt.xlabel(r'$ \varepsilon$', fontsize = 25)
    plt.ylabel(r'$u(\varepsilon)$', fontsize = 25)
    plt.legend(fontsize = 18)
    plt.grid(True)

    safe_name = sintomo.replace(" ", "_").replace("/", "-")
    filename = f"utility_{safe_name}.png"

    plt.savefig(filename, dpi=300, bbox_inches='tight')
    plt.close()

# Creo il DataFrame finale
df_result = pd.DataFrame(results)
nome_file =  'metastaes50D.csv'  # 'metastaes7D.csv' 'metastaes14D.csv' 'metastaes21D.csv' 'metastaes100D.csv'    
df_result.to_csv(nome_file, index=False)

# DEGREE AND METASTATES CORRELATION

In [ ]:
data = T_sel.to_numpy() 
Tlen, N = data.shape
win = 210
step = 10
threshold = 0.2
win_starts = np.arange(0, Tlen - win + 1, step) 
numW = len(win_starts)

# Graph creation
A = np.zeros((N, N, numW))
A_sim = np.zeros((N, N, numW))
timeIndex = np.empty(numW, dtype='datetime64[ns]')

date = df['date']

for k, start in enumerate(win_starts):
    try:
        idx = np.arange(start, start + win)
        t_center = idx[-1]
        timeIndex[k] = df.date.iloc[t_center]
        Xw = data[idx, :]
        
        std_cols = Xw.std(axis=0)
        const_cols = std_cols == 0
        if np.any(const_cols):
            Xw[:, const_cols] += 1e-8 * np.random.randn(Xw.shape[0], np.sum(const_cols))
            
        deltaW = np.diff(Xw, axis=0)
        C = np.abs(np.corrcoef(deltaW, rowvar=False))
        np.fill_diagonal(C, 0)
        
        C = np.nan_to_num(C)
        A[:, :, k] = C
    except Exception as e:
        print("Crash at k =", k, " error:", e)
            
nodi = list(T_sel.columns)

graphs = []
for k in range(len(win_starts)):
    G = nx.from_numpy_array(A[:,:,k])
    G = nx.relabel_nodes(G, dict(zip(range(len(nodi)), nodi)))
    graphs.append(G)

degree_dfs = []
betweenness_dfs = []

nodes_to_remove = ['phy_dizzy', 'phy_drymouth', 'phy_nauseous']

for G in graphs:
    G.remove_nodes_from([n for n in nodes_to_remove if n in G])

for k, G in enumerate(graphs):
    deg_dict = dict(G.degree(weight='weight'))
    bc_dict = nx.betweenness_centrality(G, weight='weight')
    
    degree_dfs.append(pd.DataFrame(deg_dict, index=[f'win_{k}']).T)
    betweenness_dfs.append(pd.DataFrame(bc_dict, index=[f'win_{k}']).T)

degree_over_time = pd.concat(degree_dfs, axis=1)
betweenness_over_time = pd.concat(betweenness_dfs, axis=1)

degree_over_time = degree_over_time.copy()
degree_over_time['mean_deg'] = degree_over_time.mean(axis=1)
betweenness_over_time = betweenness_over_time.copy()
betweenness_over_time['mean_betweenness'] = betweenness_over_time.mean(axis=1)

# Items extraction
sintomi = [
    "mood_relaxed50d", "mood_down50d", "mood_irritat50d", "mood_satisfi50d",
    "mood_lonely50d", "mood_anxious50d", "mood_enthus50d", "mood_suspic50d",
    "mood_cheerf50d", "mood_guilty50d", "mood_doubt50d", "mood_strong50d",
    "pat_restl50d", "pat_agitate50d", "pat_worry50d", "pat_concent50d",
    "se_selflike50d", "se_ashamed50d", "se_selfdoub50d", "se_handle50d",
    "phy_hungry50d", "phy_tired50d", "phy_pain50d", 
    "phy_headache50d", "phy_sleepy50d"
]

number_metastates = [
    4, 4, 3, 7, 3, 4, 3, 2, 3, 3, 4, 3, 6, 4, 3, 4, 5, 2, 4, 3, 3, 4, 2, 2, 6
]

opt_bkps = [4, 4, 5, 5, 5, 6, 5, 3, 6, 4, 3, 3, 5, 5, 3, 3, 4, 3, 4, 5, 4, 3, 5, 3, 3]

df_metastates = pd.DataFrame({
    "symptom": sintomi,
    "number_metastates": number_metastates,
    "averege degree": degree_over_time['mean_deg'],
    "averege betweenness": betweenness_over_time['mean_betweenness'],
    "optimal breakpoints": opt_bkps
})

# Correlation 
numW = degree_over_time.shape[1] - 1

s_val = []
p_val = []
corr = []
md = []
for i in range(numW):
    b_col = degree_over_time.iloc[:,i]
    y = df_metastates['number_metastates']
    md.append(np.mean(b_col))
    mask = (~b_col.isna())
    b_clean = b_col[mask]

    if b_clean.nunique() <= 1:
        rp, rs = np.nan, np.nan
    else:
        rp, pp = pearsonr(b_clean, y)
        rs, ps = spearmanr(b_clean, y)

    p_val.append(rp)
    s_val.append(rs)

# PLOT
plt.figure(figsize=(12,5))
plt.plot(timeIndex, p_val, label='Pearson')
plt.plot(timeIndex, s_val, label='Spearman', linestyle='--')
plt.xlabel('Window')
plt.ylabel('Correlation')
plt.title('Degree and Metastates correlation')
plt.legend()
plt.show()

plt.figure(figsize=(12,5))
plt.plot(timeIndex, md/max(md), label='avg degree', linestyle='-')
plt.show()

In [ ]:
# Avarege degree changing point
model = "l1"  # "l2", "rbf"
algo = rpt.BottomUp(model=model, min_size=2, jump=5).fit(np.array(md))

my_bkps = algo.predict(n_bkps=1)
cp = my_bkps[0]
cp_date = timeIndex[cp]

plt.figure(figsize = (12,6))
plt.plot(timeIndex, md/max(md))
plt.axvspan(timeIndex.min(), cp_date, color='lightblue', alpha=0.2)   # prima del changepoint
plt.axvspan(cp_date, timeIndex.max(), color='lightcoral', alpha=0.2)   # dopo il changepoint

plt.xlabel('Date', fontsize = 25)
plt.ylabel('Average degree', fontsize = 25)
plt.tick_params(axis='both', which='major', labelsize=18)
plt.grid(True)
plt.show()

In [ ]:
# Mean SCL-90 item changin point
ds['resptime_e'] = ds['resptime_e'].apply(
    lambda t: t.time() if isinstance(t, pd.Timestamp) else t
)

ds['resptime_e_td'] = ds['resptime_e'].apply(
    lambda t: pd.Timedelta(hours=t.hour, minutes=t.minute, seconds=t.second)
)
ds['date'] = pd.to_datetime(ds['date'], format="%d/%m/%y")
ds['date_time'] = ds['date'] + ds['resptime_e_td']

dep_clean = ds['dep'].dropna()
date_clean = ds.loc[dep_clean.index, 'date']# change point detection
model = "l1"  # "l2", "rbf"
algo = rpt.BottomUp(model=model, min_size=2, jump=5).fit(np.array(ds.dep.values))

my_bkps = algo.predict(n_bkps=1)
cp2 = my_bkps[0]
cp_date2 = ds.iloc[cp2]['date']

# PLOT
plt.figure(figsize=(12, 6))
plt.plot(date_clean, dep_clean, label='SCL-90-R item score')
plt.axvspan(date_clean.min(), cp_date2, color='lightblue', alpha=0.2, label='Prima del CP')
plt.axvspan(cp_date2, date_clean.max(), color='lightcoral', alpha=0.2, label='Dopo il CP')
plt.xlabel('Date', fontsize=25)
plt.ylabel('Average score on SCL-90 items', fontsize=25)
plt.tick_params(axis='both', which='major', labelsize=18)
plt.title('Average score on SCL-90 items with Change Point', fontsize=20)
plt.grid(True)
plt.legend(fontsize=14)
plt.show()